# Compositional analysis of single cell data through scCODA package 

## Preparation of packages

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import mudata as md
import muon as mu
import mudatasets as mds
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy.external as sce
from scipy import stats
from Bio import SeqIO
from matplotlib.pyplot import rc_context
import anndata as ad
import warnings
import sccoda.util
from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz

import sccoda.datasets as scd

warnings.filterwarnings("ignore")

In [ ]:
from matplotlib.patches import Patch

In [ ]:
sc.settings.verbosity = 3
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, facecolor='white', format = 'pdf', vector_friendly = True)

In [ ]:
umap_cmap = sns.blend_palette(['lightgrey', 'xkcd:sapphire'], as_cmap = True)

In [ ]:
figure = "Figure_2-3"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

## Load and prepare dataset 

In [ ]:
adata = sc.read_h5ad('h5ad/analysis_250528_f/Smed_L78-L47_20250523_Annotated.h5ad')
adata

##### **Subsetting only G2 and G1**

In [ ]:
adata.obs.columns

In [ ]:
#View the names of the sample types
adata.obs['Sample'].unique()

In [ ]:
#Subsetting G1 and G2 samples
cells_scoda = adata.obs['Sample'].isin(['G1a', 'G1b', 'G2'])
g2_g1 = adata[cells_scoda, :]

**First the contrast matrix is made defining the samples and conditions of each subset** 

In [ ]:
#G2 and G1 Contrast Matrix
cov_df_g2_g1 = pd.DataFrame({"Cond": ["G1", "G1", "G2"]},
    index=["G1a", "G1b", "G2"])
print(cov_df_g2_g1)

In [ ]:
#Retrieve the unique values in the 'annotated_names' column of the obs dataframe within the adata object
adata.obs['annotated_names'].unique()

In [ ]:
comp_data_g2_g1 = sccoda.util.cell_composition_data.from_scanpy(g2_g1, 
                                        cell_type_identifier='annotated_names', 
                                        sample_identifier='Sample',
                                        covariate_df= cov_df_g2_g1)

In [ ]:
print(comp_data_g2_g1.var) 

In [ ]:
comp_data_g2_g1

**Quick visualization from <u>scCODA</u> **

In [ ]:
with plt.rc_context({'figure.figsize': (20, 8)}):
    viz.boxplots(comp_data_g2_g1, 
                 feature_name="Cond", 
                 y_scale="log",
                 cmap='BuPu')# the scale in the y axis can be normal or log scale 
    plt.show()

## SCODA analysis

In [ ]:
model_g1 = mod.CompositionalAnalysis(comp_data_g2_g1, formula="C(Cond, Treatment('G1'))"                                )                    

In [ ]:
sim_results_g1 = model_g1.sample_hmc()

In [ ]:
sim_results_g1.summary()#results table can be extended with summary_extended() 

In [ ]:
print(sim_results_g1.credible_effects())

**Adjusting the False Discovery Rate**

- scCODA selects credible effects based on their inclusion probability. The cutoff between credible and non-credible effects depends on the desired false discovery rate (FDR).
- For this dataset we use a FDR of 0.0.5

In [ ]:
pd.options.display.max_rows=2000 
sim_results_g1.set_fdr(est_fdr=0.05)
print(sim_results_g1.credible_effects())

**Secondly, for visualizing which clusters are enriched and have credible effects, we calculate the logarithm of the odd values outcome from a simple fisher test:**

In [ ]:
def FisherTest (adata, clusteringlayer, samples_name, s1, s2):
    
    # Obtaining counts:
    cell_numbers = adata.obs[[samples_name, clusteringlayer]].groupby(samples_name)                         
    cellcounts = {}                                                                                        
    for i in adata.obs[samples_name].cat.categories:                                                       
        cellcounts[i] = cell_numbers.get_group(i)[clusteringlayer].value_counts().rename(i).sort_index()   
    counts_df = pd.DataFrame.from_dict(cellcounts)                                                         
    
    oddsdict ={}     
    pvalsdict = {}   

    for j in adata.obs[clusteringlayer].cat.categories:                                 
        cl_counts = counts_df.loc[j,[s1, s2]].to_list()                                 
        ncl_counts = (counts_df[[s1, s2]].sum() - counts_df.loc[j,[s1, s2]]).to_list()  
        oddsratio, pvalue = stats.fisher_exact([cl_counts, ncl_counts])                 
        oddsdict[j] = oddsratio                                                         
        pvalsdict[j] = pvalue                                                           
        
    counts_dict = {}   

    percs_df = counts_df[[s2, s1]] / counts_df[[s2, s1]].sum(axis = 0) * 100  
    percs_diff = percs_df[s2]-percs_df[s1]                                    
    
    fisher_df = pd.concat([pd.Series(pvalsdict),pd.Series(oddsdict), percs_diff], axis = 1, keys = ['pvals', 'odds', '% diff'])     
    
    return fisher_df, counts_df  

In [ ]:
adata.obs

In [ ]:
#1  G1
clusteringlayer = 'annotated_names'
samples_name = 'Condition'
g2 = 'G2'
g1 = 'G1'
fisher_test_g2_g1, counts_df_g1 = FisherTest(g2_g1, clusteringlayer, samples_name, g2, g1)

In [ ]:
fisher_results_g2_g1 = pd.concat([fisher_test_g2_g1, counts_df_g1], axis=1).reindex(fisher_test_g2_g1.index)

In [ ]:
fisher_results_g2_g1

In [ ]:
log2_odds_g1 = pd.DataFrame(np.log2(fisher_results_g2_g1['odds']))

In [ ]:
# get the list of enriched/depleted cell types
true_cell_types = sim_results_g1.credible_effects()[sim_results_g1.credible_effects() == True].index.get_level_values('Cell Type').unique().tolist()

In [ ]:
li_neg = log2_odds_g1[log2_odds_g1['odds'] < 0].index.to_list()
li_pos = log2_odds_g1[log2_odds_g1['odds'] > 0].index.to_list()

In [ ]:
ct_neg = [i for i in true_cell_types if i in li_neg]
ct_pos = [i for i in true_cell_types if i in li_pos]

In [ ]:
ct_neg

In [ ]:
ct_pos

In [ ]:
with plt.rc_context({'figure.figsize': (20, 15)}):
    fig, axs = plt.subplot_mosaic([
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'up_g1'],
        ['odds_g1','odds_g1', 'down_g1'],
        ['odds_g1','odds_g1', 'down_g1']],
        layout='constrained')
    
# Umap of clusters Up and Down in treatment G1
    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible enriched in G1',
               size = 15, 
               groups = ct_neg, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['up_g1'])

    sc.pl.umap(adata, color='annotated_names', legend_loc='on data', 
               legend_fontoutline = 2, 
               title= 'Credible enriched in G2',
               size = 15, 
               groups = ct_pos, 
               na_in_legend=False, na_color='#f5f5f5',
               frameon=False, show = False,
               ax = axs['down_g1'])

# Odds plot for up and down regulated clusters in G1
    value_counts_output = (-1)*(log2_odds_g1['odds'])
    categories = adata.obs['annotated_names'].unique()
    counts = value_counts_output.values
    color_array= adata.uns['annotated_names_colors']
    color_dict = dict(zip(categories, color_array))

    axs['odds_g1']
    axs['odds_g1'].set_xlabel('Clusters')
    bars= axs['odds_g1'].barh(log2_odds_g1['odds'].index, 
                       (-1)*(log2_odds_g1['odds']), 
                       edgecolor= 'black', 
                       color= [color_dict.get(category, 'gray') for category in categories]) # barh for horizontal bar plot
    axs['odds_g1'].invert_yaxis() 
    axs['odds_g1'].grid(False)
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 
    axs['odds_g1'].tick_params(axis='y', labelsize=9)
    axs['odds_g1'].grid(axis='y',color='gray', linestyle='dashed', linewidth=0.5, alpha=0.5)
    axs['odds_g1'].axvline(x=0, ymin=0, ymax=1, dashes = (2,1)) 


    #plt.show()

In [ ]:
# retrieve the neoblast score
neoblast_score = pd.DataFrame.from_dict(adata.uns['neoblast_score_annotated_names'], orient='index', columns=['neoblast_score'])
neoblast_score.index.name = 'annotated_names'

In [ ]:
neoblast_score['fisher'] = log2_odds_g1['odds'].sort_index()

In [ ]:
mask = neoblast_score.index.isin(ct_neg)
neoblast_score.loc[mask, 'sig'] = 'down'
mask = neoblast_score.index.isin(ct_pos)
neoblast_score.loc[mask, 'sig'] = 'up'
neoblast_score['sig'] = neoblast_score['sig'].fillna('no')

In [ ]:
neoblast_score['cl_size'] = (fisher_results_g2_g1['G1'] + fisher_results_g2_g1['G2']).sort_index()

In [ ]:
category_colors = dict(zip(adata.obs['annotated_names'].cat.categories, 
                           adata.uns['annotated_names_colors']))
neoblast_score['colours'] = neoblast_score.index.map(category_colors)


In [ ]:
neoblast_score

In [ ]:
plt.figure(figsize=(10, 10))

# edge style
conditions = [
    neoblast_score['sig'] == 'up',
    neoblast_score['sig'] == 'down',
    neoblast_score['sig'] == 'no'
]
edge_colors = np.select(conditions, ['#00008B', '#8B0000', 'grey'], default='grey')
line_widths = np.select(conditions, [2.5, 2.5, 0.6], default=0.8)


scatter = plt.scatter(
    x=neoblast_score['neoblast_score'].replace(0, np.nan),
    y=neoblast_score['fisher'].replace(0, np.nan),
    s=neoblast_score['cl_size'],  # bubble size
    alpha=0.8, # transparency
    color=neoblast_score['colours'],
    edgecolors = edge_colors,
    linewidths = line_widths, 
    zorder=2    
)

plt.axvline(x=0.171, color='black', linestyle='--', linewidth=1, zorder=1)
plt.axvline(x=0.258, color='black', linestyle='--',  linewidth=1, zorder=1)
plt.axhline(y=0, color='black', linewidth=0.5, zorder=1)

plt.xlabel('Neoblast score', fontsize=21)
plt.ylabel('Fisher test: log2 odd values', fontsize=21)

plt.tight_layout()
#plt.savefig( './Figure_plots/'+figure +  '/SCcoda_bubble_plot.pdf', format='pdf')
plt.show()


In [ ]:
adata.uns['Sample_colors']

In [ ]:
plt.figure(figsize=(10, 10))

# edge style
conditions = [
    neoblast_score['sig'] == 'up',
    neoblast_score['sig'] == 'down',
    neoblast_score['sig'] == 'no'
]
edge_colors = np.select(conditions, ['#b8952e', 'darkred', 'grey'], default='grey')
line_widths = np.select(conditions, [3, 3, 0.6], default=0.8)


scatter = plt.scatter(
    y=neoblast_score['neoblast_score'].replace(0, np.nan),
    x=neoblast_score['fisher'].replace(0, np.nan),
    s=neoblast_score['cl_size'],  # bubble size
    alpha=0.8, # transparency
    color=neoblast_score['colours'],
    edgecolors = edge_colors,
    linewidths = line_widths, 
    zorder=2    
)

plt.axhline(y=0.171, color='black', linestyle='--', linewidth=1, zorder=1)
plt.axhline(y=0.258, color='black', linestyle='--',  linewidth=1, zorder=1)
plt.axvline(x=0, color='black', linewidth=0.5, zorder=1)

plt.ylabel('Neoblast score', fontsize=21)
plt.xlabel('Fisher test: log2 odd values', fontsize=21)

plt.tight_layout()
plt.savefig( './Figure_plots/'+figure +  '/SCcoda_bubble_plot-2.pdf', format='pdf')
plt.show()


In [ ]:
sc.set_figure_params(figsize=(14, 6), dpi=300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'wspace': 0.3})  # reduce space between UMAPs

# ------------------ UMAP 1 -------------------
sc.pl.umap(
    adata,
    color='annotated_names',
    legend_loc='none',
    size=20,
    title= "",
    groups=ct_neg,
    na_in_legend=False,
    na_color='#f5f5f5',
    frameon=False,
    show=False,
    ax=ax1
)

all_categories = adata.obs['annotated_names'].cat.categories
all_colors = adata.uns['annotated_names_colors']

handles_neg = [Patch(color=all_colors[list(all_categories).index(g)], label=g) 
               for g in ct_neg if g in all_categories]

legend1 = ax1.legend(
    handles=handles_neg,
    loc="upper center",
    bbox_to_anchor=(0.50, 0.02),
    frameon=False,
    fontsize=10.5,
    ncol=2,
    handletextpad=0.4,
    columnspacing=0.9
)

# ------------------ UMAP 2 -------------------
sc.pl.umap(
    adata,
    color='annotated_names',
    legend_loc='none',
    size=20,
    title= "",
    groups=ct_pos,
    na_in_legend=False,
    na_color='#f5f5f5',
    frameon=False,
    show=False,
    ax=ax2
)

handles_pos = [Patch(color=all_colors[list(all_categories).index(g)], label=g)
               for g in ct_pos if g in all_categories]

legend2 = ax2.legend(
    handles=handles_pos,
    loc="upper center",
    bbox_to_anchor=(0.50, 0.02), 
    frameon=False,
    fontsize=10.5,
    ncol=3,
    handletextpad=0.4,
    columnspacing=0.9
)

#plt.savefig('./Figure_plots/'+figure + "/UMAP_SCcoda.pdf", bbox_inches="tight")
plt.show()
